### Embed PubMed journal articles into Weaviate

PubMed MultiLabel Text Classification Dataset MeSH: https://www.kaggle.com/datasets/owaiskhan9654/pubmed-multilabel-text-classification

In [23]:
!uv pip install openai

Using Python 3.12.13 environment at: C:\Github\rag\.venv
Checked 1 package in 191ms
c:\Github\rag\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
c:\Github\rag\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
c:\Github\rag\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [24]:
!uv pip install SPARQLWrapper

Using Python 3.12.13 environment at: C:\Github\rag\.venv
Checked 1 package in 35ms


In [25]:
!uv pip install --force-reinstall "weaviate-client>=3.26.7,<4.0.0"

Using Python 3.12.13 environment at: C:\Github\rag\.venv
Resolved 12 packages in 584ms
Prepared 12 packages in 18ms
Uninstalled 12 packages in 508ms
Installed 12 packages in 435ms
 ~ authlib==1.7.2
 ~ certifi==2026.6.17
 ~ cffi==2.0.0
 ~ charset-normalizer==3.4.7
 ~ cryptography==49.0.0
 ~ idna==3.18
 ~ joserfc==1.7.1
 ~ pycparser==3.0
 ~ requests==2.34.2
 ~ urllib3==2.7.0
 ~ validators==0.35.0
 ~ weaviate-client==3.26.7


In [26]:
import os
import json
from datetime import datetime

def log_experiment(approach_name, description, query_type, parameters, results, generated_text=None, filename="query_experiments.json"):
    """
    Appends an experiment run to a local tracking file for comparison.
    Now optionally captures raw LLM generated summary text.
    """
    simplified_results = []
    
    if isinstance(results, dict):
        data_payload = results.get("data", {}) or {}
        get_payload = data_payload.get("Get", {}) or data_payload.get("get", {}) or {}
        
        if get_payload:
            classname = list(get_payload.keys())[0]
            items = get_payload[classname]
            
            if isinstance(items, list):
                for item in items:
                    if item is None:
                        simplified_results.append({"title": "No Data", "id": "No ID"})
                        continue
                    title = item.get("title", "No Title")
                    additional_node = item.get("_additional")
                    weaviate_id = additional_node.get("id", "No ID") if isinstance(additional_node, dict) else "No ID"
                    simplified_results.append({"title": title, "id": weaviate_id})
            else:
                simplified_results = [{"error": f"Collection data was empty or invalid type: {type(items)}"}]
        elif 'errors' in results:
            simplified_results = [{"server_error": err.get("message", "Unknown error") for err in results.get('errors', [])}]
        else:
            simplified_results = [results]
    else:
        simplified_results = results

    # Build the log entry
    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "approach_name": approach_name,
        "description": description,
        "query_type": query_type,
        "parameters": parameters,
        "results_count": len(simplified_results) if simplified_results and "error" not in simplified_results[0] else 0,
        "results_sample": simplified_results[:5]
    }
    
    # NEW: If text was explicitly generated by an LLM, save it directly to the log entry!
    if generated_text:
        log_entry["generated_text_output"] = generated_text
    
    # Read existing logs if file exists, append, and save
    existing_data = []
    if os.path.exists(filename):
        with open(filename, "r") as f:
            try:
                existing_data = json.load(f)
            except json.JSONDecodeError:
                existing_data = []
                
    existing_data.append(log_entry)
    
    with open(filename, "w") as f:
        json.dump(existing_data, f, indent=4)
        
    print(f"💾 Experiment '{approach_name}' tracked safely to {filename}!")

In [27]:
from weaviate.util import generate_uuid5
import weaviate
#import json
import pandas as pd


In [28]:
#import pandas as pd
#Read in the pubmed data
df = pd.read_csv("PubMed Multi Label Text Classification Dataset Processed.csv")

In [29]:
#import os
#import weaviate
from dotenv import load_dotenv, find_dotenv  # Fixed the typo here!

# Load the variables from the .env file
load_dotenv(find_dotenv())

# Connect to Weaviate
client = weaviate.Client(
    url=os.getenv("WEAVIATE_URL"),
    auth_client_secret=weaviate.auth.AuthApiKey(api_key=os.getenv("WEAVIATE_API_KEY")),
    additional_headers={
        "X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")
    }
)

In [30]:
df = df[:10000]

In [31]:
import time
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# --- STEP 1: FORCE CLEAN PREVIOUS DELETIONS AND WAIT ---
target_class = "Articles"  # Check for uppercase variant
target_class_lower = "articles"  # Check for lowercase variant

for cls_name in [target_class, target_class_lower]:
    try:
        # Check if the class still registers in the schema layout
        schema = client.schema.get()
        existing_classes = [c["class"] for c in schema.get("classes", [])]
        
        if cls_name in existing_classes:
            print(f"⚠️ Collection '{cls_name}' is still taking up the slot. Forcing deletion...")
            client.schema.delete_class(cls_name)
            
            # Active Wait Loop: Give the cloud cluster up to 15 seconds to finish background cleanup
            print("⏳ Waiting for Weaviate Cloud to finish asynchronous background deletion...")
            for attempt in range(15):
                time.sleep(1)
                updated_schema = client.schema.get()
                still_there = any(c["class"] == cls_name for c in updated_schema.get("classes", []))
                if not still_there:
                    print(f"✅ Confirmed: '{cls_name}' completely cleared from cluster storage.")
                    break
    except Exception as e:
        print(f"Notice: Skip check for '{cls_name}': {e}")

# Double check that our sandbox tier is truly empty
try:
    final_check = client.schema.get().get("classes", [])
    if len(final_check) >= 1:
        print(f"🚨 Warning: Cluster slot still occupied by: {[c['class'] for c in final_check]}. Forcing a direct wipe...")
        for c in final_check:
            client.schema.delete_class(c["class"])
        time.sleep(3)  # Short safety cushion
except Exception:
    pass


# --- STEP 2: DEFINE AND CREATE THE FRESH SCHEMA ---
# We use your updated clean configuration dictionary
class_obj = {
    "class": "articles",
    "description": "A collection of articles with their abstracts, mesh terms, and URIs",
    "vectorizer": "text2vec-openai",

    "properties": [
        {"name": "title", "dataType": ["text"]},
        {"name": "abstractText", "dataType": ["text"]},
        {"name": "meshMajor", "dataType": ["text"]},
        {"name": "article_URI", "dataType": ["text"]}
    ],

    "moduleConfig": {
        "text2vec-openai": {
            "vectorizeClassName": True,
            "model": "ada",
            "type": "text"
        },
        "generative-openai": {},  # Keeps your summary capabilities active safely
        "qna-openai": {}
    }
}

print("\n🚀 Sending creation payload to empty sandbox cluster...")
try:
    client.schema.create_class(class_obj)
    print(f"🎉 Success! Collection '{class_obj['class']}' created and ready for data ingest.")
except Exception as e:
    print(f"❌ Creation failed: {e}")

⚠️ Collection 'Articles' is still taking up the slot. Forcing deletion...
⏳ Waiting for Weaviate Cloud to finish asynchronous background deletion...
✅ Confirmed: 'Articles' completely cleared from cluster storage.

🚀 Sending creation payload to empty sandbox cluster...
🎉 Success! Collection 'articles' created and ready for data ingest.


In [32]:
#import logging
import numpy as np
from weaviate.util import generate_uuid5

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# Replace infinity values with NaN and then fill NaN values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna('', inplace=True)

# Convert columns to string type
df['Title'] = df['Title'].astype(str)
df['abstractText'] = df['abstractText'].astype(str)

# Log the data types
logging.info(f"Title column type: {df['Title'].dtype}")
logging.info(f"abstractText column type: {df['abstractText'].dtype}")

with client.batch(
    batch_size=10,  # Specify batch size
    num_workers=2,   # Parallelize the process
) as batch:
    for index, row in df.iterrows():
        try:
            question_object = {
                "title": row.Title,
                "abstractText": row.abstractText,
            }
            batch.add_data_object(
                question_object,
                class_name="articles",
                uuid=generate_uuid5(question_object)
            )
        except Exception as e:
            logging.error(f"Error processing row {index}: {e}")


2026-06-20 19:58:02,120 INFO Title column type: str
2026-06-20 19:58:02,124 INFO abstractText column type: str


In [33]:
client.query.aggregate("articles").with_meta_count().do()

{'data': {'Aggregate': {'Articles': [{'meta': {'count': 9997}}]}}}

In [34]:
# 1. Define your description
approach = "Basic Collection Fetch"
desc = "Pulls a raw, un-filtered sample of your data so you can verify that your batch uploader worked, check your properties, and inspect your auto-generated object IDs."
params = {"additional": "id", "limit": 2}

# 2. Run your query
response = (
    client.query
    .get("articles", ["title", "abstractText"])
    .with_additional(["id"])
    .with_limit(2)
    .do()
)

# 3. Log it!
log_experiment(approach, desc, "Vector Batch Load Check", params, res)

# 4. FIXED SAFETY CHECK: Look for both lowercase and Weaviate's auto-capitalized GraphQL response key
data_payload = response.get("data", {}) or {}
get_payload = data_payload.get("Get", {}) or {}

# This handles Weaviate's internal string transformation gracefully!
articles_list = get_payload.get("articles") or get_payload.get("Articles")

if articles_list:
    print(f"\n🎉 Success! Retrieved {len(articles_list)} sample records from Weaviate:\n" + "="*50)
    
    for index, article in enumerate(articles_list, 1):
        title = article.get("title", "No Title Found")
        abstract = article.get("abstractText", "No Abstract Provided")
        weaviate_id = article.get("_additional", {}).get("id", "No ID Generated")
        
        print(f"\n📄 Article {index}: “{title}”")
        print(f"🆔 UUID: {weaviate_id}")
        print(f"📝 Abstract:\n{abstract}")
        print("-" * 50)  # Adds a visual separator between articles
else:
    print("❌ Query execution failed or returned no records.")
    if "errors" in response:
        print("\n🚨 Server Error Message from Weaviate:")
        for error in response["errors"]:
            print(f"- {error.get('message')}")
    else:
        print("\nFull Raw Response Structure for Analysis:")
        import json
        print(json.dumps(response, indent=2))

NameError: name 'res' is not defined

In [ ]:
import json

# 1. Define your experiment metadata and search parameters
approach = "Semantic Vector Search"
desc = "Performs a nearText semantic vector search to find the top 10 articles conceptually closest to 'Mouth Neoplasms'."
params = {
    "concepts": ["Mouth Neoplasms"], 
    "limit": 10, 
    "additional": ["id"]
}

# 2. Execute the semantic query
response = (
    client.query
    .get("articles", ["title", "abstractText"])
    .with_additional(params["additional"])
    .with_near_text({"concepts": params["concepts"]})
    .with_limit(params["limit"])
    .do()
)

# 3. Log the run details to your tracking log file
log_experiment(approach, desc, "Vector NearText Search", params, response)

# 4. Extract data safely using Weaviate's capitalized GraphQL response signature
data_payload = response.get("data", {}) or {}
get_payload = data_payload.get("Get", {}) or {}
articles_list = get_payload.get("articles") or get_payload.get("Articles")

if articles_list:
    print(f"\n🎉 Success! Retrieved {len(articles_list)} semantic matches for '{params['concepts'][0]}':\n" + "="*60)
    
    for index, article in enumerate(articles_list, 1):
        title = article.get("title", "No Title Found")
        abstract = article.get("abstractText", "No Abstract Provided")
        weaviate_id = article.get("_additional", {}).get("id", "No ID Generated")
        
        # Display short snippet of the abstract for space efficiency in the terminal
        short_abstract = abstract[:150] + "..." if len(abstract) > 150 else abstract
        
        print(f"🔍 Result {index}: “{title}”")
        print(f"   UUID: {weaviate_id}")
        print(f"   Snippet: {short_abstract}")
        print("-" * 60)
else:
    print("❌ Search returned no results or hit a system exception.")
    if "errors" in response:
        print("\n🚨 Server Error Message from Weaviate:")
        for error in response["errors"]:
            print(f"- {error.get('message')}")

💾 Experiment 'Semantic Search' successfully logged to query_experiments.json!

🎉 Success! Retrieved 10 sample records from Weaviate:

📄 Article 1: “Gingival metastasis as first sign of multiorgan dissemination of epithelioid malignant mesothelioma.”
🆔 UUID: a7690f03-66b9-5d17-b765-8c6eb21f99c8
📝 Abstract:
INTRODUCTION: Metastatic malignant mesothelioma to the oral cavity is extremely rare. They are more common in the jaw bones than the soft tissue. Occurrence of the malignant disease typically carries an average survival rate of 9-12 monthsMETHODS: : Thirteen patients underwent neoadjuvant chemotherapy and radical pleurectomy decortication, followed by radiotherapy from August 2012 to September 2013. Patients were followed up with computed tomography of the chest and the abdomen every 3 months. All patients were followed up until February 2014.RESULTS: In January 2014, 11 patients were still alive with a median survival of 11 months, eight patients developed a recurrence and two patien

In [ ]:
import json

# 1. Define your experiment metadata and search parameters
approach = "Object-to-Object Similarity Search"
desc = "Uses an existing article's UUID vector embedding as a baseline to find the top 10 conceptually similar articles."
params = {
    "source_id": "a7690f03-66b9-5d17-b765-8c6eb21f99c8", 
    "limit": 10, 
    "additional": ["distance"]
}

# 2. Execute the query (Using your standardized client and lowercase collection name)
response = (
    client.query
    .get("articles", ["title", "abstractText"])
    .with_near_object({
        "id": params["source_id"]
    })
    .with_limit(params["limit"])
    .with_additional(params["additional"])
    .do()
)

# 3. Log the run details to your experiment tracking file
log_experiment(approach, desc, "Vector NearObject Search", params, response)

# 4. Extract data safely using Weaviate's capitalized GraphQL response signature
data_payload = response.get("data", {}) or {}
get_payload = data_payload.get("Get", {}) or {}
articles_list = get_payload.get("articles") or get_payload.get("Articles")

if articles_list:
    print(f"\n🎉 Success! Found {len(articles_list)} articles similar to Object ID: {params['source_id']}\n" + "="*75)
    
    for index, article in enumerate(articles_list, 1):
        title = article.get("title", "No Title Found")
        
        # Pull out the mathematical distance score from the metadata layer
        meta = article.get("_additional", {})
        distance_score = meta.get("distance", "N/A")
        
        print(f"🔗 Match {index} | (Cosine Distance: {distance_score})")
        print(f"   Title: “{title}”")
        print("-" * 75)
else:
    print("❌ Search returned no results. Check if the baseline UUID exists in your dataset.")
    if "errors" in response:
        print("\n🚨 Server Error Message from Weaviate:")
        for error in response["errors"]:
            print(f"- {error.get('message')}")

💾 Experiment 'Object-to-Object Similarity Search' successfully logged to query_experiments.json!

🎉 Success! Found 10 articles similar to Object ID: a7690f03-66b9-5d17-b765-8c6eb21f99c8
🔗 Match 1 | (Cosine Distance: 1.1920929e-07)
   Title: “Gingival metastasis as first sign of multiorgan dissemination of epithelioid malignant mesothelioma.”
---------------------------------------------------------------------------
🔗 Match 2 | (Cosine Distance: 0.12507892)
   Title: “Survival patterns for malignant mesothelioma: the SEER experience.”
---------------------------------------------------------------------------
🔗 Match 3 | (Cosine Distance: 0.12555015)
   Title: “Extended Pleurectomy and Decortication for Malignant Pleural Mesothelioma Is an Effective and Safe Cytoreductive Surgery in the Elderly.”
---------------------------------------------------------------------------
🔗 Match 4 | (Cosine Distance: 0.12759769)
   Title: “Metastatic neuroblastoma in the mandible. Report of a case.”
--

In [ ]:
import json

# 1. Define your experiment metadata and search parameters
approach = "RAG Generative Search"
desc = "Finds the closest matching medical case study via vector search and uses an LLM to explain the title into a layperson definition."
params = {
    "concept_query": "Gingival metastasis as first sign of multiorgan dissemination of epithelioid malignant mesothelioma",
    "prompt_instruction": "Please explain this article {title} like you would to someone without a medical degree.",
    "limit": 1
}

# 2. Execute the Generative RAG Query
response = (
    client.query
    .get("articles", ["title", "abstractText"]) # Standardized to lowercase 'articles'
    .with_near_text({"concepts": [params["concept_query"]]})
    .with_generate(single_prompt=params["prompt_instruction"])
    .with_limit(params["limit"])
    .do()
)

# 3. Log the run details to your tracking log file
log_experiment(approach, desc, "RAG Generative Search", params, response)

# 4. Extract data safely using Weaviate's capitalized GraphQL response signature
data_payload = response.get("data", {}) or {}
get_payload = data_payload.get("Get", {}) or {}
articles_list = get_payload.get("articles") or get_payload.get("Articles")

if articles_list:
    print(f"\n🎉 Success! Retrieval and Generation Complete:\n" + "="*75)
    
    for index, article in enumerate(articles_list, 1):
        title = article.get("title", "No Title Found")
        
        # Pull out the dynamic generation text blocks from Weaviate's special _additional metadata node
        meta = article.get("_additional", {}) or {}
        generate_node = meta.get("generate", {}) or {}
        
        # 'singleResult' holds the output text generated by OpenAI for this specific row entry
        llm_explanation = generate_node.get("singleResult", "❌ LLM generation failed or was empty.")
        
        print(f"📖 Source Article Match: “{title}”")
        print(f"\n🤖 AI Layperson Explanation:\n{llm_explanation}")
        print("="*75)
else:
    print("❌ Query returned no data or failed during execution.")
    if "errors" in response:
        print("\n🚨 Server Error Message from Weaviate:")
        for error in response["errors"]:
            print(f"- {error.get('message')}")

💾 Experiment 'RAG Generative Search' successfully logged to query_experiments.json!

🎉 Success! Retrieval and Generation Complete:
📖 Source Article Match: “Gingival metastasis as first sign of multiorgan dissemination of epithelioid malignant mesothelioma.”

🤖 AI Layperson Explanation:
Here’s a plain-language explanation of what that article is about and why it matters.

What the article reports, in one sentence
- Doctors found that a new lump on a patient’s gum (the gingiva) turned out to be a spread (metastasis) of malignant mesothelioma — a rare cancer that usually starts in the lining of the lungs or abdomen. The gum lesion was the first clue that the cancer had already spread to several organs.

What malignant mesothelioma is
- Mesothelioma is a rare cancer that starts in the thin lining (the mesothelium) that covers organs such as the lungs (pleura) or abdomen (peritoneum). It is often linked to long-ago exposure to asbestos.
- “Epithelioid” is one subtype of mesothelioma. It beh

In [ ]:
import json

# ==============================================================================
# 1. DEFINE EXPERIMENT METADATA & SEARCH PARAMETERS
# ==============================================================================
approach = "Grouped RAG Search"
desc = (
    "Retrieves the top 3 semantic matches for 'Mouth Neoplasms' and uses an LLM "
    "to synthesize them into a single, unified bulleted summary."
)
params = {
    "concept_query": "Mouth Neoplasms",
    "grouped_task_instruction": (
        "Summarize the key information here in bullet points. "
        "Make it understandable to someone without a medical degree."
    ),
    "limit": 3
}


# ==============================================================================
# 2. EXECUTE THE GROUPED GENERATIVE RAG QUERY
# ==============================================================================
print("🔍 Querying Weaviate and running Grouped LLM Generation...")
response = (
    client.query
    .get("articles", ["title", "abstractText"]) # Standardized to lowercase 'articles'
    .with_near_text({"concepts": [params["concept_query"]]})
    .with_limit(params["limit"])
    .with_generate(grouped_task=params["grouped_task_instruction"])
    .do()
)


# ==============================================================================
# 3. LOG THE EXPERIMENT
# ==============================================================================
# Calls your updated, bulletproof logging helper function
log_experiment(approach, desc, "Grouped RAG Generative Search", params, response)


# ==============================================================================
# 4. PARSE AND DISPLAY THE RESULTS BEAUTIFULLY
# ==============================================================================
# Safely handle Weaviate's auto-capitalized GraphQL response keys
data_payload = response.get("data", {}) or {}
get_payload = data_payload.get("Get", {}) or {}
articles_list = get_payload.get("articles") or get_payload.get("Articles")

if articles_list:
    print("\n" + "="*80)
    print("🎉 SUCCESS: RETRIEVAL AND GROUPED SYNTHESIS COMPLETE")
    print("="*80)
    
    # In a grouped task, the final unified output is stored in the metadata layer
    # of the VERY FIRST item in the returned articles array.
    first_article_meta = articles_list[0].get("_additional", {}) or {}
    generate_node = first_article_meta.get("generate", {}) or {}
    
    # Check for any LLM provider-side API or context tokens errors
    generation_error = generate_node.get("error")
    unified_summary = generate_node.get("groupedResult")
    
    if generation_error:
        print(f"❌ LLM Generation Error: {generation_error}")
    elif unified_summary:
        print("🤖 AI Synthesized Summary (Across all source papers):")
        print(unified_summary)
        print("="*80)
        
        # Print out the supporting background references used in the prompt context
        print("\n📚 Source Papers Utilized for this Summary Context:")
        for idx, article in enumerate(articles_list, 1):
            print(f"  {idx}. {article.get('title')}")
    else:
        print("⚠️ Query executed, but the 'groupedResult' field came back empty.")
else:
    print("\n❌ Query returned no data or failed during execution.")
    if "errors" in response:
        print("\n🚨 Server Error Message from Weaviate Cloud:")
        for error in response["errors"]:
            print(f"- {error.get('message')}")

🔍 Querying Weaviate and running Grouped LLM Generation...
💾 Experiment 'Grouped RAG Search' tracked safely to query_experiments.json!

🎉 SUCCESS: RETRIEVAL AND GROUPED SYNTHESIS COMPLETE
🤖 AI Synthesized Summary (Across all source papers):
- Study 1 — Gingival (gum) metastasis from mesothelioma
  - Metastatic spread of mesothelioma (a cancer usually in the lining of the lungs) to the mouth is extremely rare, and when it happens it is more often in jaw bone than in soft tissue like the gums.
  - In a group of 13 patients treated with chemotherapy, surgery (removing the lung lining), and radiation, the typical survival was about 9–12 months; at one-year follow-up 11 patients were alive, eight had cancer come back, and two had died.
  - One 68-year-old man developed a gum lump one year after surgery; biopsy showed it was a metastasis from his original mesothelioma and it was the first sign that the cancer had spread to multiple organs. He received more chemotherapy and local radiation to 

### Turn metadata into a KG

In [ ]:
from rdflib import Graph, RDF, RDFS, Namespace, URIRef, Literal
from rdflib.namespace import SKOS, XSD
import pandas as pd
import urllib.parse
import random
from datetime import datetime, timedelta

# Create a new RDF graph
g = Graph()

# Define namespaces
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
prefixes = {
    'schema': schema,
    'ex': ex,
    'skos': SKOS,
    'xsd': XSD
}
for p, ns in prefixes.items():
    g.bind(p, ns)

# Define classes and properties
Article = URIRef(ex.Article)
MeSHTerm = URIRef(ex.MeSHTerm)
g.add((Article, RDF.type, RDFS.Class))
g.add((MeSHTerm, RDF.type, RDFS.Class))

title = URIRef(schema.name)
abstract = URIRef(schema.description)
date_published = URIRef(schema.datePublished)
access = URIRef(ex.access)

g.add((title, RDF.type, RDF.Property))
g.add((abstract, RDF.type, RDF.Property))
g.add((date_published, RDF.type, RDF.Property))
g.add((access, RDF.type, RDF.Property))

# Function to clean and parse MeSH terms
def parse_mesh_terms(mesh_list):
    if pd.isna(mesh_list):
        return []
    return [term.strip().replace(' ', '_') for term in mesh_list.strip("[]'").split(',')]

# Function to create a valid URI
def create_valid_uri(base_uri, text):
    if pd.isna(text):
        return None
    sanitized_text = urllib.parse.quote(text.strip().replace(' ', '_').replace('"', '').replace('<', '').replace('>', '').replace("'", "_"))
    return URIRef(f"{base_uri}/{sanitized_text}")

# Function to generate a random date within the last 5 years
def generate_random_date():
    start_date = datetime.now() - timedelta(days=5*365)
    random_days = random.randint(0, 5*365)
    return start_date + timedelta(days=random_days)

# Function to generate a random access value between 1 and 10
def generate_random_access():
    return random.randint(1, 10)

# Load your DataFrame here
# df = pd.read_csv('your_data.csv')

# Loop through each row in the DataFrame and create RDF triples
for index, row in df.iterrows():
    article_uri = create_valid_uri("http://example.org/article", row['Title'])
    if article_uri is None:
        continue
    
    # Add Article instance
    g.add((article_uri, RDF.type, Article))
    g.add((article_uri, title, Literal(row['Title'], datatype=XSD.string)))
    g.add((article_uri, abstract, Literal(row['abstractText'], datatype=XSD.string)))
    
    # Add random datePublished and access
    random_date = generate_random_date()
    random_access = generate_random_access()
    g.add((article_uri, date_published, Literal(random_date.date(), datatype=XSD.date)))
    g.add((article_uri, access, Literal(random_access, datatype=XSD.integer)))
    
    # Add MeSH Terms
    mesh_terms = parse_mesh_terms(row['meshMajor'])
    for term in mesh_terms:
        term_uri = create_valid_uri("http://example.org/mesh", term)
        if term_uri is None:
            continue
        
        # Add MeSH Term instance
        g.add((term_uri, RDF.type, MeSHTerm))
        g.add((term_uri, RDFS.label, Literal(term.replace('_', ' '), datatype=XSD.string)))
        
        # Link Article to MeSH Term
        g.add((article_uri, schema.about, term_uri))

# Serialize the graph to a file (optional)
g.serialize(destination='ontology.ttl', format='turtle')


KeyboardInterrupt: 

### Semantic search using KG

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

def get_concept_triples_for_term(term):
    sparql = SPARQLWrapper("https://id.nlm.nih.gov/mesh/sparql")
    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    PREFIX mesh: <http://id.nlm.nih.gov/mesh/>

    SELECT ?subject ?p ?pLabel ?o ?oLabel
    FROM <http://id.nlm.nih.gov/mesh>
    WHERE {{
        ?subject rdfs:label "{term}"@en .
        ?subject ?p ?o .
        FILTER(CONTAINS(STR(?p), "concept"))
        OPTIONAL {{ ?p rdfs:label ?pLabel . }}
        OPTIONAL {{ ?o rdfs:label ?oLabel . }}
    }}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    
    triples = set()  # Using a set to avoid duplicate entries
    for result in results["results"]["bindings"]:
        obj_label = result.get("oLabel", {}).get("value", "No label")
        triples.add(obj_label)
    
    # Add the term itself to the list
    triples.add(term)
    
    return list(triples)  # Convert back to a list for easier handling

def get_narrower_concepts_for_term(term):
    sparql = SPARQLWrapper("https://id.nlm.nih.gov/mesh/sparql")
    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    PREFIX mesh: <http://id.nlm.nih.gov/mesh/>

    SELECT ?narrowerConcept ?narrowerConceptLabel
    WHERE {{
        ?broaderConcept rdfs:label "{term}"@en .
        ?narrowerConcept meshv:broaderDescriptor ?broaderConcept .
        ?narrowerConcept rdfs:label ?narrowerConceptLabel .
    }}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    
    concepts = set()  # Using a set to avoid duplicate entries
    for result in results["results"]["bindings"]:
        subject_label = result.get("narrowerConceptLabel", {}).get("value", "No label")
        concepts.add(subject_label)
    
    return list(concepts)  # Convert back to a list for easier handling

def get_all_narrower_concepts(term, depth=2, current_depth=1):
    # Create a dictionary to store the terms and their narrower concepts
    all_concepts = {}

    # Initial fetch for the primary term
    narrower_concepts = get_narrower_concepts_for_term(term)
    all_concepts[term] = narrower_concepts
    
    # If the current depth is less than the desired depth, fetch narrower concepts recursively
    if current_depth < depth:
        for concept in narrower_concepts:
            # Recursive call to fetch narrower concepts for the current concept
            child_concepts = get_all_narrower_concepts(concept, depth, current_depth + 1)
            all_concepts.update(child_concepts)
    
    return all_concepts

# Fetch alternative names and narrower concepts
term = "Mouth Neoplasms"
alternative_names = get_concept_triples_for_term(term)
all_concepts = get_all_narrower_concepts(term, depth=4)  # Adjust depth as needed

# Output alternative names
print("Alternative names:", alternative_names)
print()

# Output narrower concepts
for broader, narrower in all_concepts.items():
    print(f"Broader concept: {broader}")
    print(f"Narrower concepts: {narrower}")
    print("---")


Alternative names: ['Cancer of Mouth', 'Mouth Neoplasms']

Broader concept: Mouth Neoplasms
Narrower concepts: ['Palatal Neoplasms', 'Tongue Neoplasms', 'Lip Neoplasms', 'Leukoplakia, Oral', 'Salivary Gland Neoplasms', 'Gingival Neoplasms']
---
Broader concept: Palatal Neoplasms
Narrower concepts: []
---
Broader concept: Tongue Neoplasms
Narrower concepts: []
---
Broader concept: Lip Neoplasms
Narrower concepts: []
---
Broader concept: Leukoplakia, Oral
Narrower concepts: ['Leukoplakia, Hairy']
---
Broader concept: Leukoplakia, Hairy
Narrower concepts: []
---
Broader concept: Salivary Gland Neoplasms
Narrower concepts: ['Sublingual Gland Neoplasms', 'Submandibular Gland Neoplasms', 'Parotid Neoplasms']
---
Broader concept: Sublingual Gland Neoplasms
Narrower concepts: []
---
Broader concept: Submandibular Gland Neoplasms
Narrower concepts: []
---
Broader concept: Parotid Neoplasms
Narrower concepts: []
---
Broader concept: Gingival Neoplasms
Narrower concepts: []
---


In [ ]:
def flatten_concepts(concepts_dict):
    flat_list = []

    def recurse_terms(term_dict):
        for term, narrower_terms in term_dict.items():
            flat_list.append(term)
            if narrower_terms:
                recurse_terms(dict.fromkeys(narrower_terms, []))  # Use an empty dict to recurse
    
    recurse_terms(concepts_dict)
    return flat_list

# Flatten the concepts dictionary
flat_list = flatten_concepts(all_concepts)

In [ ]:
#Convert the MeSH terms to URI
def convert_to_mesh_uri(term):
    formatted_term = term.replace(" ", "_").replace(",", "_").replace("-", "_")
    return URIRef(f"http://example.org/mesh/_{formatted_term}_")


# Convert terms to URIs
mesh_terms = [convert_to_mesh_uri(term) for term in flat_list]

In [ ]:
import os
import json
from datetime import datetime

# ==============================================================================
# 1. SET METADATA FOR TRACKING LOG
# ==============================================================================
approach = "Knowledge Graph SPARQL Search"
desc = "Queries a local RDF graph via SPARQL to find and rank articles by the number of matching interconnected MeSH terms."
# Captures the specific list of nodes you are filtering by
params = {
    "target_mesh_terms": [str(m) for m in mesh_terms],
    "limit": 3
}


# ==============================================================================
# 2. EXECUTE GRAPH TRAVERSAL & SYNTHESIS
# ==============================================================================
article_data = {}

for mesh_term in mesh_terms:
    results = g.query(query, initBindings={'meshTerm': mesh_term})

    for row in results:
        article_uri = str(row['article'])

        if article_uri not in article_data:
            article_data[article_uri] = {
                'title': str(row['title']),
                'abstract': str(row['abstract']),
                'datePublished': str(row['datePublished']),
                'access': str(row['access']),
                'meshTerms': set()
            }
        article_data[article_uri]['meshTerms'].add(str(row['meshTerm']))

# Sort by count intersection size
ranked_articles = sorted(
    article_data.items(),
    key=lambda item: len(item[1]['meshTerms']),
    reverse=True
)

top_3_articles = ranked_articles[:3]


# ==============================================================================
# 3. CONVERT RESULTS & TRACK LOGS
# ==============================================================================
# Standardize output format so it plays nice with your tracking json layout
formatted_results_for_log = []
for uri, data in top_3_articles:
    formatted_results_for_log.append({
        "title": data['title'],
        "id": uri  # Uses the unique RDF URI string as the identifier anchor
    })

# Write safely to the centralized query tracking pipeline
log_experiment(approach, desc, "SPARQL Knowledge Graph Query", params, formatted_results_for_log)


# ==============================================================================
# 4. PRINT RESULTS VISUALLY
# ==============================================================================
print(f"\n🎉 Success! Displaying Top {len(top_3_articles)} Graph-Intersected Articles:\n" + "="*80)
for index, (article_uri, data) in enumerate(top_3_articles, 1):
    print(f"📄 Article {index}: “{data['title']}”")
    print(f"🔗 URI: {article_uri}")
    print(f"🎯 Matching MeSH Terms Count: {len(data['meshTerms'])}")
    print(f"📝 Abstract Snippet: {data['abstract'][:200]}...")
    print("📋 All Mapped Terms:")
    for term in data['meshTerms']:
        print(f"  - {term}")
    print("-" * 80)

💾 Experiment 'Knowledge Graph SPARQL Search' tracked safely to query_experiments.json!

🎉 Success! Displaying Top 3 Graph-Intersected Articles:
📄 Article 1: “Myoepithelioma of minor salivary gland origin. Light and electron microscopical study.”
🔗 URI: http://example.org/article/Myoepithelioma_of_minor_salivary_gland_origin._Light_and_electron_microscopical_study.
🎯 Matching MeSH Terms Count: 2
📝 Abstract Snippet: A gingival tumor that invaded the anterior maxilla was removed from a 14-year-old boy and studied by light and electron microscopy. The tumor was composed exclusively of myoepithelial cells and appear...
📋 All Mapped Terms:
  - http://example.org/mesh/_Gingival_Neoplasms_
  - http://example.org/mesh/_Salivary_Gland_Neoplasms_
--------------------------------------------------------------------------------
📄 Article 2: “NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition.”
🔗 URI: http://example.org/article/NDRG1_

### Similarity search using a KG

In [39]:
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, RDFS, Namespace, SKOS
import urllib.parse
import os
import json
from datetime import datetime

# ==============================================================================
# 1. INITIALIZE ONTOLOGY NAMESPACES
# ==============================================================================
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
rdfs = Namespace('http://www.w3.org/2000/01/rdf-schema#')

# ==============================================================================
# 2. METRIC MATHEMATICAL UTILITIES
# ==============================================================================
def jaccard_similarity(set1, set2):
    """Calculates intersection over union score matrix between two sets."""
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    similarity = len(intersection) / len(union) if len(union) != 0 else 0
    return similarity, intersection

# ==============================================================================
# 3. VOLATILE MEMORY GRAPH LAZY-LOADING
# ==============================================================================
if 'g' not in locals():
    print("Loading ontology.ttl...")
    g = Graph()
    g.parse('ontology.ttl', format='turtle')
    print("Graph loaded successfully!")
else:
    print("Graph already loaded in memory.")

# ==============================================================================
# 4. STRUCTURAL DATA URI MAPPING HELPERS
# ==============================================================================
def get_article_uri(title):
    cleaned_title = title.strip().replace("“", "").replace("”", "").replace('"', "")
    underscored_title = cleaned_title.replace(" ", "_")
    return URIRef(f"http://example.org/article/{underscored_title}")

def get_mesh_terms(article_uri):
    query = """
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>

    SELECT ?meshTerm
    WHERE {
      ?article schema:about ?meshTerm .
      ?meshTerm a ex:MeSHTerm .
      FILTER (?article = <""" + str(article_uri) + """>)
    }
    """
    results = g.query(query)
    return {str(row['meshTerm']) for row in results}

# ==============================================================================
# 5. MODIFIED SEARCH ROUTINE WITH RESULT RETRIEVAL EXTRACTOR
# ==============================================================================
def find_similar_articles(title):
    article_uri = get_article_uri(title)
    mesh_terms_given_article = get_mesh_terms(article_uri)
    
    print(f"Found {len(mesh_terms_given_article)} MeSH terms for the target article.")
    if not mesh_terms_given_article:
        print("⚠️ Warning: No MeSH terms found for this article title. Check your URI formatting!")
        return []

    mesh_terms_other_articles = {}
    print("Running high-speed graph intersection query...")
    
    for term in mesh_terms_given_article:
        query = f"""
        PREFIX schema: <http://schema.org/>
        PREFIX ex: <http://example.org/>

        SELECT ?article
        WHERE {{
          ?article schema:about <{term}> .
        }}
        """
        results = g.query(query)
        for row in results:
            other_article = str(row['article'])
            if other_article != str(article_uri):
                if other_article not in mesh_terms_other_articles:
                    mesh_terms_other_articles[other_article] = set()
                mesh_terms_other_articles[other_article].add(str(term))

    # Calculate Jaccard metrics across intersections
    similarities = {}
    overlapping_terms = {}
    for article, mesh_terms in mesh_terms_other_articles.items():
        similarity, overlap = jaccard_similarity(mesh_terms_given_article, mesh_terms)
        similarities[article] = similarity
        overlapping_terms[article] = overlap

    # Sort results and return the raw extraction arrays
    top_similar_articles = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:15]
    
    return top_similar_articles, overlapping_terms

# ==============================================================================
# 6. RUN AND LOG EXPERIMENT PIPELINE
# ==============================================================================
article_title = "Gingival metastasis as first sign of multiorgan dissemination of epithelioid malignant mesothelioma."

# Execute lookups
search_results = find_similar_articles(article_title)

if search_results:
    top_articles, total_overlaps = search_results
    
    # Define your structured logging configurations
    approach = "Graph Jaccard Similarity"
    desc = "Finds articles similar to a target resource based on overlapping MeSH term mappings within an RDF graph using Jaccard indices."
    params = {
        "target_article_title": article_title,
        "metric_type": "Jaccard Intersection over Union",
        "limit": 15
    }
    
    # Map graph elements into compatible schemas for tracking logs
    formatted_results_for_log = []
    for article_uri, similarity_score in top_articles:
        formatted_results_for_log.append({
            "title": f"Graph Entity Match (Jaccard Score: {similarity_score:.4f})",
            "id": article_uri
        })
        
    # Append execution snapshot details straight into the centralized logger file
    log_experiment(approach, desc, "Graph Network Jaccard Search", params, formatted_results_for_log)

    # Output details beautifully directly inside the notebook environment
    print(f"\n🎉 Success! Top {len(top_articles)} Graph-Similar Entities Identified:")
    print("="*85)
    for article, similarity in top_articles:
        print(f"🔗 Article URI: {article}")
        print(f"   📊 Jaccard Score: {similarity:.4f}")
        print(f"   🎯 Overlapping Shared Labels: {len(total_overlaps[article])} matches found")
        print("-" * 85)
else:
    print("❌ Analysis pipeline terminated because no entity connections were matched.")

Graph already loaded in memory.
Found 0 MeSH terms for the target article.
⚠️ Warning: No MeSH terms found for this article title. Check your URI formatting!
❌ Analysis pipeline terminated because no entity connections were matched.


In [ ]:
from rdflib import URIRef

# Function to retrieve title and abstract for a given article URI
def get_article_details(article_uri):
    query = """
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>

    SELECT ?title ?abstract ?datePublished ?access
    WHERE {{
      <{}> a ex:Article ;
            schema:name ?title ;
            schema:description ?abstract ;
            schema:datePublished ?datePublished ;
            ex:access ?access .
    }}
    """.format(article_uri)

    # Execute the query
    results = g.query(query)

    # Process results
    for row in results:
        title = row['title']
        abstract = row['abstract']
        date_published = row['datePublished']
        access = row['access']
        return title, abstract, date_published, access

    # Return None if no results found
    return None, None, None, None

# Example usage
article_uri = "http://example.org/article/CT_scan_cerebral_hemispheric_asymmetries%3A_predictors_of_recovery_from_aphasia."  # Use the URI as a string
title, abstract, date_published, access = get_article_details(article_uri)

if title and abstract:
    print(f"Title: {title}")
    print(f"Abstract: {abstract}")
    print(f"Date Published: {date_published}")
    print(f"Access: {access}")
else:
    print("No article found with the given URI.")


Title: CT scan cerebral hemispheric asymmetries: predictors of recovery from aphasia.
Abstract: Individual variations in anatomic cerebral asymmetries have been linked with specific neurodevelopmental processes, with patterns of cognitive ability, and with recovery from focal brain damage. The present study investigated relationships between cerebral asymmetries and recovery from aphasia. Aphasic patients (N = 25) were assessed for language recovery for 1 year poststroke, and linear measurements of cerebral asymmetries were performed on CT scans. Increasing left occipital width asymmetry was associated with faster rate of language recovery and with higher final language scores during the first year poststroke. There was, moreover, a tendency for increasing left occipital width asymmetry to be associated with less initial impairment. It is hypothesized that those aspects of neural organization conferring better premorbid language skills are the same factors conferring greater recovery o

### RAG using a KG

In [ ]:
# Function to combine titles and abstracts
def combine_abstracts(top_3_articles):
    combined_text = "".join(
        [f"Title: {data['title']} Abstract: {data['abstract']}" for article_uri, data in top_3_articles]
    )
    return combined_text

# Combine abstracts from the top 3 articles
combined_text = combine_abstracts(top_3_articles)
print(combined_text)


Title: Myoepithelioma of minor salivary gland origin. Light and electron microscopical study. Abstract: A gingival tumor that invaded the anterior maxilla was removed from a 14-year-old boy and studied by light and electron microscopy. The tumor was composed exclusively of myoepithelial cells and appeared to be malignant. By light microscopy, the tumor appeared to be a poorly differentiated epithelial neoplasm of undetermined origin; however, electron microscopical examination showed myoepithelial differentiation, indicative of a salivary gland origin. To our knowledge, the present case represents the only confirmed myoepithelioma that shows features indicative of malignant potential. Myoepitheliomas may be related to mixed tumors of salivary glands.Title: NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition. Abstract: Regional metastasis is the single most important prognostic factor in oral squamous cell carcinoma (OSCC)

In [ ]:
import os
import pandas as pd
import openai
from dotenv import load_dotenv, find_dotenv

# 1. Look for the .env file and load its variables into the environment
load_dotenv(find_dotenv(), override=True)

# 2. Extract the key safely using os.getenv
# (Using the clean stripping technique to avoid trailing whitespace errors)
raw_key = os.getenv("OPENAI_API_KEY") or ""
openai.api_key = raw_key.strip().replace('"', '').replace("'", "")

print("OpenAI API Key successfully loaded from configuration file.")

OpenAI API Key successfully loaded from configuration file.


In [ ]:
import openai

# Initialize client
openai_client = openai.OpenAI()

def generate_and_log_summary(text_context):
    approach = "Direct LLM Synthesis"
    desc = "Sends a raw concatenated string of medical articles directly to GPT-4o-mini for layperson summary synthesis."
    params = {
        "model": "gpt-4o-mini",
        "temperature": 0.3,
        "max_tokens": 1000,
        "context_character_length": len(text_context)
    }

    print("🤖 Dispatching text payload to OpenAI Completion Engine...")
    
    try:
        response = openai_client.chat.completions.create(
            model=params["model"],
            messages=[
                {
                    "role": "user",
                    "content": f"Summarize the key information here in bullet points. Make it understandable to someone without a medical degree:\n\n{text_context}"
                }
            ],
            max_tokens=params["max_tokens"],
            temperature=params["temperature"]
        )
        
        raw_summary = response.choices[0].message.content.strip()
        lines = [line.strip() for line in raw_summary.split('\n') if line.strip()]
        formatted_summary = '\n'.join(lines)
        
        # Define reference log layout
        formatted_results_for_log = [{
            "title": f"LLM Generation Summary ({params['model']})",
            "id": f"char_len_{params['context_character_length']}"
        }]
        
        # FIXED: Passing formatted_summary as the 6th argument (generated_text parameter)
        log_experiment(
            approach_name=approach, 
            description=desc, 
            query_type="Standalone ChatCompletion Summary", 
            parameters=params, 
            results=formatted_results_for_log,
            generated_text=formatted_summary # This will now write it to the file!
        )
        
        return formatted_summary

    except Exception as e:
        print(f"\n❌ API execution encountered an error: {str(e)}")
        return None

# Execute process
if 'combined_text' in locals() or 'combined_text' in globals():
    summary = generate_and_log_summary(combined_text)
    if summary:
        print("\n" + "="*80)
        print("🎉 SUCCESS: PARSED AI LAYPERSON SUMMARY")
        print("="*80)
        print(summary)
        print("="*80)

### Step 3: use a vector-powered knowledge graph to test data retrieval

Create a column to store URIs for each article

In [ ]:
# Function to create a valid URI
def create_valid_uri(base_uri, text):
    if pd.isna(text):
        return None
    # Encode text to be used in URI
    sanitized_text = urllib.parse.quote(text.strip().replace(' ', '_').replace('"', '').replace('<', '').replace('>', '').replace("'", "_"))
    return URIRef(f"{base_uri}/{sanitized_text}")

# Add a new column to the DataFrame for the article URIs
df['Article_URI'] = df['Title'].apply(lambda title: create_valid_uri("http://example.org/article", title))


Test that the URI works - we are able to find all mesh terms associated with a given uri

In [ ]:
from rdflib import Graph, Namespace, URIRef

# Assuming your RDF graph (g) is already loaded

# Define namespaces
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
rdfs = Namespace('http://www.w3.org/2000/01/rdf-schema#')

def get_mesh_terms_for_article(graph, article_uri):
    # Define the SPARQL query using the article URI
    query = f"""
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?meshTermLabel
    WHERE {{
      <{article_uri}> a ex:Article ;
               schema:about ?meshTerm .
      ?meshTerm rdfs:label ?meshTermLabel .
    }}
    """
    
    # Execute the query
    results = graph.query(query)
    
    # Extract the MeSH terms from the results
    mesh_terms = [str(row['meshTermLabel']) for row in results]
    
    return mesh_terms

# Example usage with the provided URI
article_uri = "http://example.org/article/Expression_of_p53_and_coexistence_of_HPV_in_premalignant_lesions_and_in_cervical_cancer."
mesh_terms_example = get_mesh_terms_for_article(g, article_uri)

# Output the results
print(f"MeSH terms associated with the article '{article_uri}':")
for term in mesh_terms_example:
    print(term)


MeSH terms associated with the article 'http://example.org/article/Expression_of_p53_and_coexistence_of_HPV_in_premalignant_lesions_and_in_cervical_cancer.':
DNA Probes
HPV'
'DNA
Viral'
'Female'
'Humans'
'Immunohistochemistry'
'Papillomaviridae'
'Tumor Suppressor Protein p53'
'Uterine Cervical Dysplasia'
'Uterine Cervical Neoplasms


Create the schema for new embedding which includes mesh tags and URI

In [ ]:
class_obj = {
    # Class definition
    "class": "articles_with_abstracts_and_URIs",

    # Property definitions
    "properties": [
        {
            "name": "title",
            "dataType": ["text"],
        },
        {
            "name": "abstractText",
            "dataType": ["text"],
        },
        {
            "name": "meshMajor",
            "dataType": ["text"],
        },
        {
            "name": "Article_URI",
            "dataType": ["text"],
        },
    ],

    # Specify a vectorizer
    "vectorizer": "text2vec-openai",

    # Module settings
    "moduleConfig": {
       "text2vec-openai": {
            "vectorizeClassName": True,
            "model": "ada",  # Updated to standard formal model name
            "type": "text"
        }
    },
}

In [ ]:
# 1. Check what is currently blocking your 1-slot free tier sandbox
current_schema = client.schema.get()
existing_classes = [c["class"] for c in current_schema.get("classes", [])]

# 2. Clear out the old collection
for old_class in existing_classes:
    print(f"Deleting conflicting collection to free up sandbox slot: {old_class}...")
    client.schema.delete_class(old_class)

# 3. Create your new 'articles_with_abstracts_and_URIs' class fresh
client.schema.create_class(class_obj)
print(f"Success! Collection '{class_obj['class']}' created and ready for data.")

Deleting conflicting collection to free up sandbox slot: Articles...
Success! Collection 'articles_with_abstracts_and_URIs' created and ready for data.


In [ ]:
import logging
import numpy as np
from weaviate.util import generate_uuid5

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# Clean dataset rules
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna('', inplace=True)

# Convert types safely
df['Title'] = df['Title'].astype(str)
df['abstractText'] = df['abstractText'].astype(str)
df['meshMajor'] = df['meshMajor'].astype(str)
df['pmid'] = df['pmid'].astype(str)

# --- DEFINING A WEAVIATE BATCH ERROR CALLBACK ---
def check_batch_result(results):
    """
    Inspects server-side return statuses for every object in a batch 
    and logs hidden validation or token issues.
    """
    if results is not None:
        for result in results:
            if 'result' in result and 'errors' in result['result']:
                if 'error' in result['result']['errors']:
                    for error in result['result']['errors']['error']:
                        logging.error(f" Weaviate Server Error: {error['message']}")

print(f"Total rows to upload: {len(df)}")
print("Starting safe fixed-batch single-threaded upload to Weaviate...")

# FIXED CONFIGURATION: Fixed sizes with callback error-trapping handles
with client.batch.configure(
    batch_size=100,               # Stable standard sizing payload for abstract lengths
    dynamic=False,                # Turn off auto-throttling sleeps to prevent artificial deadlocks
    num_workers=1,                # Preserve single-threading stability on Windows local environments
    callback=check_batch_result   # Direct injection server error parser
) as batch:
    
    for index, row in df.iterrows():
        try:
            generated_uri = f"http://example.org/article/{row.pmid}"
            
            question_object = {
                "title": row.Title,
                "abstractText": row.abstractText,
                "meshMajor": row.meshMajor,
                "article_URI": generated_uri,
            }
            
            # NOTE: Verify your class name matches exactly what was built in your class_obj definition cell!
            batch.add_data_object(
                question_object,
                class_name="Articles_with_abstracts_and_URIs",
                uuid=generate_uuid5(question_object)
            )
            
            # Real-time console progress logger updates
            if (index + 1) % 100 == 0 or (index + 1) == len(df):
                print(f"🔄 Flushed and pushed batch group: {index + 1}/{len(df)} articles processed...")
                
        except Exception as e:
            logging.error(f"Frontend row processing exception at index {index}: {e}")

print("🎉 Fixed batch upload process concluded successfully!")

Total rows to upload: 10000
Starting safe fixed-batch single-threaded upload to Weaviate...
🔄 Flushed and pushed batch group: 100/10000 articles processed...
🔄 Flushed and pushed batch group: 200/10000 articles processed...
🔄 Flushed and pushed batch group: 300/10000 articles processed...
🔄 Flushed and pushed batch group: 400/10000 articles processed...
🔄 Flushed and pushed batch group: 500/10000 articles processed...
🔄 Flushed and pushed batch group: 600/10000 articles processed...
🔄 Flushed and pushed batch group: 700/10000 articles processed...
🔄 Flushed and pushed batch group: 800/10000 articles processed...
🔄 Flushed and pushed batch group: 900/10000 articles processed...
🔄 Flushed and pushed batch group: 1000/10000 articles processed...
🔄 Flushed and pushed batch group: 1100/10000 articles processed...
🔄 Flushed and pushed batch group: 1200/10000 articles processed...
🔄 Flushed and pushed batch group: 1300/10000 articles processed...
🔄 Flushed and pushed batch group: 1400/10000 a

### Semantic search with vectorized KG

In [ ]:
import json

# ==============================================================================
# 1. DEFINE EXPERIMENT METADATA & SEARCH PARAMETERS
# ==============================================================================
approach = "Semantic Vector Search"
desc = "Performs a nearText semantic vector search to identify the top 10 articles conceptually closest to 'mouth neoplasms'."
params = {
    "concept_query": "mouth neoplasms",
    "limit": 10,
    "target_collection": "Articles_with_abstracts_and_URIs"
}

# ==============================================================================
# 2. EXECUTE THE VECTOR SEARCH QUERY
# ==============================================================================
print(f"🔍 Searching Weaviate '{params['target_collection']}' index for semantic matches...")

try:
    response = (
        client.query
        .get(params["target_collection"], ["title", "abstractText", "meshMajor", "article_URI"])
        .with_additional(["id"])
        .with_near_text({"concepts": [params["concept_query"]]})
        .with_limit(params["limit"])
        .do()
    )

    # ==========================================================================
    # 3. LOG THE EXPERIMENT RUN
    # ==========================================================================
    # Pass the native response directly. The tracking pipeline will automatically
    # map out the class name fields and append them safely to your JSON log.
    log_experiment(
        approach_name=approach,
        description=desc,
        query_type="Vector NearText Search",
        parameters=params,
        results=response
    )

    # ==========================================================================
    # 4. PRINT AND FORMAT DISPLAY OUTPUT
    # ==========================================================================
    data_payload = response.get("data", {}) or {}
    get_payload = data_payload.get("Get", {}) or {}
    
    # Check both lowercase and uppercase keys to remain safe across versions
    articles_list = get_payload.get("Articles_with_abstracts_and_URIs") or get_payload.get("articles_with_abstracts_and_URIs")

    if articles_list:
        print("\n" + "="*85)
        print(f"🎉 SUCCESS: FOUND {len(articles_list)} SEMANTIC MATCHES")
        print("="*85)
        
        for index, article in enumerate(articles_list, 1):
            title = article.get("title", "No Title Found")
            uri = article.get("article_URI", "No URI Found")
            
            # Extract the UUID securely from the metadata node
            meta = article.get("_additional", {}) or {}
            v_id = meta.get("id", "No UUID")
            
            print(f" [{index}] Match Score Rank Rank")
            print(f"    📄 Title: {title}")
            print(f"    🆔 UUID:  {v_id}")
            print(f"    🔗 URI:   {uri}")
            print("-" * 85)
    else:
        print("\n⚠️ Query finished, but returned zero matches. Check if data ingestion completed.")
        if "errors" in response:
            print(f"🚨 Server Error Notice: {json.dumps(response['errors'], indent=2)}")

except Exception as e:
    print(f"\n❌ Query failed on connection layer execution: {str(e)}")

🔍 Searching Weaviate 'Articles_with_abstracts_and_URIs' index for semantic matches...
💾 Experiment 'Semantic Vector Search' tracked safely to query_experiments.json!

🎉 SUCCESS: FOUND 10 SEMANTIC MATCHES
 [1] Match Score Rank Rank
    📄 Title: Gingival metastasis as first sign of multiorgan dissemination of epithelioid malignant mesothelioma.
    🆔 UUID:  179fa21e-740a-513e-95c0-44ff4c4a0c3f
    🔗 URI:   http://example.org/article/25157778
-------------------------------------------------------------------------------------
 [2] Match Score Rank Rank
    📄 Title: Mandibular pseudocarcinomatous hyperplasia.
    🆔 UUID:  5b558001-b04d-5100-958c-4248ce020cf2
    🔗 URI:   http://example.org/article/10931233
-------------------------------------------------------------------------------------
 [3] Match Score Rank Rank
    📄 Title: Angiocentric Centrofacial Lymphoma as a Challenging Diagnosis in an Elderly Man.
    🆔 UUID:  60b1ad4e-5396-5c76-8ced-b2c333012c49
    🔗 URI:   http://example.or

In [ ]:
# Extract article URIs
article_uris = [article["article_URI"] for article in response["data"]["Get"]["Articles_with_abstracts_and_URIs"]]

In [ ]:
article_uris

['http://example.org/article/25157778',
 'http://example.org/article/10931233',
 'http://example.org/article/30926768',
 'http://example.org/article/1061918',
 'http://example.org/article/26944382',
 'http://example.org/article/48374',
 'http://example.org/article/24202525',
 'http://example.org/article/12642957',
 'http://example.org/article/184921',
 'http://example.org/article/10764053']

In [10]:
# ==============================================================================
# 1. DEFENSIVE CHECK: Ensure we actually have candidate titles to search for
# ==============================================================================
# Try to fallback to your hardcoded article title if your previous step's array was empty
if 'top_articles' not in locals() or len(top_articles) == 0:
    print("⚠️ 'top_articles' array was empty. Falling back to default target article...")
    # Using the primary paper from your earlier experiments as a backup anchor
    candidate_titles = ["Gingival metastasis as first sign of multiorgan dissemination of epithelioid malignant mesothelioma."]
else:
    # Safely extract titles from your prior Weaviate/Jaccard steps depending on their layout
    candidate_titles = []
    for item in top_articles:
        if isinstance(item, dict):
            candidate_titles.append(item.get("title"))
        elif isinstance(item, tuple) and isinstance(item[0], str):
            # If it's a tuple from the Jaccard entity list, we extract the URI or string name
            candidate_titles.append(item[0].split('/')[-1].replace('_', ' '))

print(f"📋 Candidate titles being passed to Knowledge Graph: {len(candidate_titles)}")

# ==============================================================================
# 2. RUN BULLETPROOF SPARQL TEMPLATE
# ==============================================================================
# We modify the date condition so it handles both date-math AND string-matching fallbacks
final_matches = []

for title_item in candidate_titles:
    if not title_item:
        continue
        
    templated_query = """
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>
    
    SELECT ?article ?title ?datePublished ?abstract
    WHERE {
      ?article a ex:Article ;
               schema:name ?title ;
               schema:datePublished ?datePublished ;
               schema:description ?abstract .
               
      # Filter for title match case-insensitively to prevent spacing mismatches
      FILTER(CONTAINS(LCASE(STR(?title)), LCASE(\"\"\"""" + title_item.strip() + """\"\"\")))
    }
    """
    
    try:
        results = g.query(templated_query)
        
        for row in results:
            date_str = str(row['datePublished'])
            
            # SAFE FALLBACK DATE FILTERING: Inspecting the string directly
            # Change this condition or remove it entirely if your dataset is pre-2023!
            if date_str >= "2020-01-01": 
                final_matches.append({
                    "uri": str(row['article']),
                    "title": str(row['title']),
                    "date": date_str,
                    "abstract": str(row['abstract'])
                })
    except Exception as e:
        print(f"❌ Sub-query failed for title '{title_item[:30]}...': {e}")

# ==============================================================================
# 3. CONVERT, LOG, AND PRINT RESULTS
# ==============================================================================
if final_matches:
    print(f"\n🎉 Success! Found {len(final_matches)} matching articles in graph space:")
    print("="*85)
    for idx, match in enumerate(final_matches[:3], 1):
        print(f" [{idx}] {match['title']}")
        print(f"    📅 Published: {match['date']}")
        print(f"    🔗 Graph URI: {match['uri']}")
        print("-" * 85)
        
    # Log the successful retrieval transaction
    log_experiment(
        approach_name="Graph Date Filter",
        description="Filters graph entities by string title matching and explicit publication dates.",
        query_type="SPARQL Temporal Filter",
        parameters={"date_threshold": "2020-01-01", "candidate_count": len(candidate_titles)},
        results=[{"title": m["title"], "id": m["uri"]} for m in final_matches]
    )
else:
    print("\n❌ Still found 0 matches. Your graph triples likely contain older data.")
    print("💡 Suggestion: Remove or lower the date filter completely to see your historical papers!")

⚠️ 'top_articles' array was empty. Falling back to default target article...
📋 Candidate titles being passed to Knowledge Graph: 1
❌ Sub-query failed for title 'Gingival metastasis as first s...': name 'g' is not defined

❌ Still found 0 matches. Your graph triples likely contain older data.
💡 Suggestion: Remove or lower the date filter completely to see your historical papers!


### Similarity search with vectorized KG

In [27]:
meta_check = (
    client.query
    .aggregate("Articles_with_abstracts_and_URIs")
    .with_meta_count()
    .do()
)

# Extract total documented records
total_records = meta_check['data']['Aggregate']['Articles_with_abstracts_and_URIs'][0]['meta']['count']

# Sample to check if vectors exist on a standard get lookup
vector_sample = (
    client.query
    .get("Articles_with_abstracts_and_URIs", ["title"])
    .with_additional(["vector"])
    .with_limit(5)
    .do()
)

print(f"Total scalar rows inside collection: {total_records}")

Total scalar rows inside collection: 9997


In [14]:
import json

# ==============================================================================
# 1. DYNAMIC STEP: Fetch the first record with a valid vector from your cluster
# ==============================================================================
print("🔍 Sampling your cluster to retrieve a verified vector anchor node...")
sampler_response = (
    client.query
    .get("Articles_with_abstracts_and_URIs", ["title"])
    .with_additional(["id", "vector"])
    .with_limit(1)
    .do()
)

try:
    sample_list = sampler_response['data']['Get']['Articles_with_abstracts_and_URIs']
    if not sample_list:
        raise ValueError("The collection is completely empty. Run your batch ingest cell first!")
    
    # Extract the true active UUID from the database
    valid_anchor_id = sample_list[0]['_additional']['id']
    anchor_title = sample_list[0]['title']
    print(f"✅ Found valid anchor target: “{anchor_title[:50]}...”")
    print(f"📌 Target UUID: {valid_anchor_id}\n")

except (KeyError, TypeError, ValueError) as e:
    print(f"❌ Target verification failed: {e}")
    print("Ensure your schema is actively tracking vectors and data has been indexed successfully.")
    valid_anchor_id = None


# ==============================================================================
# 2. RUN NEIGHBORHOOD SEARCH USING THE VERIFIED ANCHOR
# ==============================================================================
if valid_anchor_id:
    approach = "Vector NearObject Neighborhood"
    desc = f"Retrieves a cluster of 50 documents surrounding a verified active article node ({valid_anchor_id}) using vector proximity coordinates."
    params = {
        "anchor_object_id": valid_anchor_id,
        "limit": 50,
        "metric_requested": "Cosine Distance"
    }

    print(f"📡 Resolving vector neighborhood for anchor ID: {params['anchor_object_id']}...")

    try:
        response = (
            client.query
            .get("Articles_with_abstracts_and_URIs", ["title", "abstractText", "meshMajor", "article_URI"])
            .with_near_object({
                "id": params["anchor_object_id"]
            })
            .with_limit(params["limit"])
            .with_additional(["distance"])
            .do()
        )

        # Log the successful transaction to query_experiments.json
        log_experiment(
            approach_name=approach,
            description=desc,
            query_type="Vector NearObject Cluster",
            parameters=params,
            results=response
        )

        # Parse and display the results layout
        data_payload = response.get("data", {}) or {}
        get_payload = data_payload.get("Get", {}) or {}
        articles_list = get_payload.get("Articles_with_abstracts_and_URIs") or get_payload.get("articles_with_abstracts_and_URIs")

        if articles_list:
            print("\n" + "="*85)
            print(f"🎉 SUCCESS: MAPPED {len(articles_list)} NEIGHBORHOOD NODES")
            print("="*85)
            
            print("🔝 TOP 3 CLOSEST SEMANTIC MATCHES:")
            for index, article in enumerate(articles_list[:3], 1):
                title = article.get("title", "No Title")
                meta = article.get("_additional", {}) or {}
                distance = meta.get("distance", 0.0)
                print(f"  {index}. Distance Score: {distance:.4f} -> “{title}”")
                
            print("\n📊 NEIGHBORHOOD DISTANCE DISTRIBUTION SUMMARY:")
            all_distances = [float(a.get("_additional", {}).get("distance", 0)) for a in articles_list]
            print(f"   • Closest Neighbor Distance: {min(all_distances):.4f}")
            print(f"   • Furthest Boundary Distance: {max(all_distances):.4f}")
            print("="*85)
        else:
            print("\n⚠️ NearObject query executed but returned zero items.")
            if "errors" in response:
                print(f"🚨 Server Error Detail: {json.dumps(response['errors'], indent=2)}")

    except Exception as e:
        print(f"\n❌ Execution pipeline stalled: {str(e)}")

🔍 Sampling your cluster to retrieve a verified vector anchor node...
✅ Found valid anchor target: “Molecular modeling of BAD complex resided in a mit...”
📌 Target UUID: 0002f1ba-9941-5a38-a32e-d706c5a5787f

📡 Resolving vector neighborhood for anchor ID: 0002f1ba-9941-5a38-a32e-d706c5a5787f...
💾 Experiment 'Vector NearObject Neighborhood' tracked safely to query_experiments.json!

🎉 SUCCESS: MAPPED 50 NEIGHBORHOOD NODES
🔝 TOP 3 CLOSEST SEMANTIC MATCHES:
  1. Distance Score: 0.0000 -> “Molecular modeling of BAD complex resided in a mitochondrion integrating glycolysis and apoptosis.”
  2. Distance Score: 0.0997 -> “Rac1 inhibits apoptosis in human lymphoma cells by stimulating Bad phosphorylation on Ser-75.”
  3. Distance Score: 0.1168 -> “Synergistic induction of apoptosis in human leukemia cells (U937) exposed to bryostatin 1 and the proteasome inhibitor lactacystin involves dysregulation of the PKC/MAPK cascade.”

📊 NEIGHBORHOOD DISTANCE DISTRIBUTION SUMMARY:
   • Closest Neighbor Dis

In [35]:
# Assuming response is the data structure with your articles
article_uris = [URIRef(article["article_URI"]) for article in response["data"]["Get"]["Articles_with_abstracts_and_URIs"]]

KeyError: 'Articles_with_abstracts_and_URIs'

In [16]:
from rdflib import URIRef

# Constructing the SPARQL query with a FILTER for the article URIs
query = """
PREFIX schema: <http://schema.org/>
PREFIX ex: <http://example.org/>

SELECT ?article ?title ?abstract ?datePublished ?access ?meshTerm
WHERE {
  ?article a ex:Article ;
           schema:name ?title ;
           schema:description ?abstract ;
           schema:datePublished ?datePublished ;
           ex:access ?access ;
           schema:about ?meshTerm .

  ?meshTerm a ex:MeSHTerm .

  # Filter to include only articles from the list of URIs
  FILTER (?article IN (%s))
}
"""


# Convert the list of URIRefs into a string suitable for SPARQL
article_uris_string = ", ".join([f"<{str(uri)}>" for uri in article_uris])

# Insert the article URIs into the query
query = query % article_uris_string

# Dictionary to store articles and their associated MeSH terms
article_data = {}

# Run the query for each MeSH term
for mesh_term in mesh_terms:
    results = g.query(query, initBindings={'meshTerm': mesh_term})

    # Process results
    for row in results:
        article_uri = row['article']

        if article_uri not in article_data:
            article_data[article_uri] = {
                'title': row['title'],
                'abstract': row['abstract'],
                'datePublished': row['datePublished'],
                'access': row['access'],
                'meshTerms': set()
            }

        # Add the MeSH term to the set for this article
        article_data[article_uri]['meshTerms'].add(str(row['meshTerm']))

# Rank articles by the number of matching MeSH terms
ranked_articles = sorted(
    article_data.items(),
    key=lambda item: len(item[1]['meshTerms']),
    reverse=True
)


# Output results
for article_uri, data in ranked_articles:
    print(f"Title: {data['title']}")
    print(f"Abstract: {data['abstract']}")
    print("MeSH Terms:")
    for mesh_term in data['meshTerms']:
        print(f"  - {mesh_term}")
    print()


NameError: name 'mesh_terms' is not defined

In [ ]:
# List to store the URIs of the ranked articles
ranked_article_uris = [URIRef(article_uri) for article_uri, data in ranked_articles]

In [1]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, SKOS

# Assuming your RDF graph (g) is already loaded

# Define namespaces
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
rdfs = Namespace('http://www.w3.org/2000/01/rdf-schema#')

def filter_articles_by_access(graph, article_uris, access_values):
    # Construct the SPARQL query with a dynamic VALUES clause
    uris_str = " ".join(f"<{uri}>" for uri in article_uris)
    query = f"""
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?article ?title ?abstract ?datePublished ?access ?meshTermLabel
    WHERE {{
      VALUES ?article {{ {uris_str} }}
      
      ?article a ex:Article ;
               schema:name ?title ;
               schema:description ?abstract ;
               schema:datePublished ?datePublished ;
               ex:access ?access ;
               schema:about ?meshTerm .
      ?meshTerm rdfs:label ?meshTermLabel .
      
      FILTER (?access IN ({", ".join(map(str, access_values))}))
    }}
    """
    
    # Execute the query
    results = graph.query(query)
    
    # Extract the details for each article
    results_dict = {}
    for row in results:
        article_uri = str(row['article'])
        if article_uri not in results_dict:
            results_dict[article_uri] = {
                'title': str(row['title']),
                'abstract': str(row['abstract']),
                'date_published': str(row['datePublished']),
                'access': str(row['access']),
                'mesh_terms': []
            }
        results_dict[article_uri]['mesh_terms'].append(str(row['meshTermLabel']))
    
    return results_dict

access_values = [3,5,7]
filtered_articles = filter_articles_by_access(g, ranked_article_uris, access_values)

# Output the results
for uri, details in filtered_articles.items():
    print(f"Article URI: {uri}")
    print(f"Title: {details['title']}")
    print(f"Abstract: {details['abstract']}")
    print(f"Date Published: {details['date_published']}")
    print(f"Access: {details['access']}")
    print()


NameError: name 'g' is not defined

### RAG with a vectorized KG

In [ ]:
response = (
    client.query
    .get("Articles_with_abstracts_and_URIs", ["title", "abstractText",'article_URI','meshMajor'])
    .with_near_text({"concepts": ["therapies for mouth neoplasms"]})
    .with_limit(3)
    .with_generate(grouped_task="Summarize the key information here in bullet points. Make it understandable to someone without a medical degree.")
    .do()
)

print(response["data"]["Get"]["Articles_with_abstracts_and_URIs"][0]["_additional"]["generate"]["groupedResult"])

In [ ]:
# Extract article URIs
article_uris = [article["article_URI"] for article in response["data"]["Get"]["Articles_with_abstracts_and_URIs"]]

# Function to filter the response for only the given URIs
def filter_articles_by_uri(response, article_uris):
    filtered_articles = []
    
    articles = response['data']['Get']['Articles_with_abstracts_and_URIs']
    for article in articles:
        if article['article_URI'] in article_uris:
            filtered_articles.append(article)
    
    return filtered_articles

# Filter the response
filtered_articles = filter_articles_by_uri(response, article_uris)

# Output the filtered articles
print("Filtered articles:")
for article in filtered_articles:
    print(f"Title: {article['title']}")
    print(f"URI: {article['article_URI']}")
    print(f"Abstract: {article['abstractText']}")
    print(f"MeshMajor: {article['meshMajor']}")
    print("---")

## RAG on vectorized knowledge graph with filters

In [40]:
response = (
    client.query
    .get("articles_with_abstracts_and_URIs", ["title", "abstractText", "meshMajor", "article_URI"])
    .with_additional(["id"])
    .with_near_text({"concepts": ["therapies for mouth neoplasms"]})
    .with_limit(20)
    .do()
)

# Assuming response is the data structure with your articles
article_uris = [URIRef(article["article_URI"]) for article in response["data"]["Get"]["Articles_with_abstracts_and_URIs"]]

ConnectionError: Query was not successful.

In [30]:
from rdflib import URIRef

# Constructing the SPARQL query with a FILTER for the article URIs
query = """
PREFIX schema: <http://schema.org/>
PREFIX ex: <http://example.org/>

SELECT ?article ?title ?abstract ?datePublished ?access ?meshTerm
WHERE {
  ?article a ex:Article ;
           schema:name ?title ;
           schema:description ?abstract ;
           schema:datePublished ?datePublished ;
           ex:access ?access ;
           schema:about ?meshTerm .

  ?meshTerm a ex:MeSHTerm .

  # Filter to include only articles from the list of URIs
  FILTER (?article IN (%s))
}
"""


# Convert the list of URIRefs into a string suitable for SPARQL
article_uris_string = ", ".join([f"<{str(uri)}>" for uri in article_uris])

# Insert the article URIs into the query
query = query % article_uris_string

# Dictionary to store articles and their associated MeSH terms
article_data = {}

# Run the query for each MeSH term
for mesh_term in mesh_terms:
    results = g.query(query, initBindings={'meshTerm': mesh_term})

    # Process results
    for row in results:
        article_uri = row['article']

        if article_uri not in article_data:
            article_data[article_uri] = {
                'title': row['title'],
                'abstract': row['abstract'],
                'datePublished': row['datePublished'],
                'access': row['access'],
                'meshTerms': set()
            }

        # Add the MeSH term to the set for this article
        article_data[article_uri]['meshTerms'].add(str(row['meshTerm']))

# Rank articles by the number of matching MeSH terms
ranked_articles = sorted(
    article_data.items(),
    key=lambda item: len(item[1]['meshTerms']),
    reverse=True
)


# Output results
for article_uri, data in ranked_articles:
    print(f"Title: {data['title']}")
    print(f"Abstract: {data['abstract']}")
    print("MeSH Terms:")
    for mesh_term in data['meshTerms']:
        print(f"  - {mesh_term}")
    print()


NameError: name 'mesh_terms' is not defined

In [ ]:
# Function to combine titles and abstracts
def combine_abstracts(ranked_articles):
    combined_text = "".join(
        [f"Title: {data['title']} Abstract: {data['abstract']}" for article_uri, data in ranked_articles]
    )
    return combined_text

# Combine abstracts from the top 3 articles
combined_text = combine_abstracts(ranked_articles)
print(combined_text)


In [ ]:
# Function to combine titles and abstracts into one chunk of text
def combine_abstracts(ranked_articles):
    combined_text = " ".join(
        [f"Title: {data['title']} Abstract: {data['abstract']}" for _, data in ranked_articles]
    )
    return combined_text

# Combine abstracts from the filtered articles
combined_text = combine_abstracts(ranked_articles)
print(combined_text)


In [ ]:
# Generate and print the summary
summary = generate_summary(combined_text)
print(summary)
